# UAV C2 — Improved Research Protocol (LOMO, 1-second windows)

## Purpose
This notebook is a **new, controlled follow-up experiment**. It loads the already-extracted feature CSVs; it does not re-read or re-extract PCAPs.

The priorities are:
1. Validate window-level labels without assuming `attack_id` is the label.
2. Audit run-length imbalance, zero-packet windows, constant and exact duplicate features.
3. Compare raw numeric features against fold-wise feature pruning.
4. Compare ordinary window-weighted training against equal-total-weight-per-run training.
5. Evaluate with **Leave-One-Mission-Out (LOMO)** and report results separately for each held-out mission.

**Do not expect this notebook to guarantee higher scores.** Its purpose is to test whether evaluation is better controlled and whether the model is less dominated by long runs.

### Important protocol rules
- `y` is the target. Do not include `y`, `attack_id`, `mission_id`, `run_id`, `channel_id`, `seed`, `condition_group`, timestamp, or window size in the feature matrix.
- Do not force `y = 1` from `attack_id == "T4_DoS"`. The feature-extraction logic uses attack-event overlap at window level; retain that convention only after verifying the CSV was created with that version.
- Feature removal is learned from each **outer training fold only**.
- Do not tune the decision threshold or hyperparameters on the held-out mission.
- Report both pooled-window metrics and metrics weighted so each run contributes equally.

In [9]:
from pathlib import Path
import hashlib
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, balanced_accuracy_score,
    average_precision_score, roc_auc_score, confusion_matrix
)

SEED = 2026
PRIMARY_WINDOW = 1
ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/uav_c2_research_improved")
OUT_DIR.mkdir(parents=True, exist_ok=True)

required = ["features_1s.csv", "features_2s.csv", "features_5s.csv"]
candidate_dirs = [ROOT] + [p for p in ROOT.rglob("*") if p.is_dir()]
feature_dir = next(
    (d for d in candidate_dirs if all((d / name).is_file() for name in required)),
    None
)
if feature_dir is None:
    raise FileNotFoundError(
        "Không tìm thấy đủ features_1s.csv, features_2s.csv, features_5s.csv trong /kaggle/input."
    )

# Read only the primary 1-second file to keep the experiment focused.
df = pd.read_csv(feature_dir / "features_1s.csv")
required_cols = {
    "y", "mission_id", "run_id", "attack_id", "window_start_epoch_s", "window_size_s"
}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_cols)}")

df["y"] = pd.to_numeric(df["y"], errors="raise").astype(int)
df["window_size_s"] = pd.to_numeric(df["window_size_s"], errors="raise")
if set(df["y"].dropna().unique()) - {0, 1}:
    raise ValueError(f"y không phải nhãn nhị phân 0/1: {sorted(df['y'].unique())}")

print("Feature folder:", feature_dir)
print("Rows × columns:", df.shape)
print("Missions:", sorted(df["mission_id"].astype(str).unique()))
print("Runs:", df["run_id"].nunique())
print("Window size values:", sorted(df["window_size_s"].dropna().unique()))
print("Label counts:", df["y"].value_counts().sort_index().to_dict())

Feature folder: /kaggle/input/datasets/nguyen273mmb/uav-c2-pcap
Rows × columns: (164517, 66)
Missions: ['F1', 'F4', 'F7']
Runs: 120
Window size values: [np.int64(1)]
Label counts: {0: 92857, 1: 71660}


## A. Label audit and run-length audit

This section checks `y` against the run-level `attack_id`, but it **does not automatically repair** differences. If `attack_overlap_fraction` is present, it also verifies the documented 50% overlap rule. If that field was intentionally stripped from the backup, consult the original event intervals and extractor version.

In [10]:
print("=" * 90)
print("A1. y × attack_id")
print(pd.crosstab(df["y"], df["attack_id"], margins=True, dropna=False))

if "attack_overlap_fraction" in df.columns:
    frac = pd.to_numeric(df["attack_overlap_fraction"], errors="coerce")
    expected_y = np.where(
        df["attack_id"].astype(str).eq("T0_Benign"),
        0,
        np.where(df["attack_id"].astype(str).eq("T4_DoS"), (frac >= 0.5).astype(int), np.nan)
    )
    valid = pd.notna(expected_y)
    mismatch_n = int((df.loc[valid, "y"].to_numpy() != np.asarray(expected_y)[valid]).sum())
    print("Mismatches against window-overlap rule (T0_Benign => 0; T4_DoS => overlap >= 0.5):", mismatch_n)
else:
    print(
        "NOTE: attack_overlap_fraction is not present in this CSV. "
        "The 533 T4_DoS/y=0 cases from the earlier audit cannot be verified from this feature table alone. "
        "Do not relabel them from attack_id; verify the exact extractor version/event timestamps."
    )

run_audit = (
    df.groupby(["run_id", "mission_id", "attack_id"], dropna=False)
      .agg(
          n_windows=("y", "size"),
          n_positive=("y", "sum"),
          positive_rate=("y", "mean"),
          first_window=("window_start_epoch_s", "min"),
          last_window=("window_start_epoch_s", "max"),
          zero_packet_windows=("packet_count", lambda s: int((pd.to_numeric(s, errors="coerce") == 0).sum()))
      )
      .reset_index()
)
run_audit["span_seconds_approx"] = (
    pd.to_numeric(run_audit["last_window"], errors="coerce")
    - pd.to_numeric(run_audit["first_window"], errors="coerce") + 1
)
run_audit.sort_values("n_windows", ascending=False).to_csv(
    OUT_DIR / "run_audit.csv", index=False
)
print("\nTop 15 longest runs by number of windows:")
print(run_audit.sort_values("n_windows", ascending=False).head(15).to_string(index=False))

print("\nRows and positive rate by mission:")
print(
    df.groupby("mission_id")["y"]
      .agg(n_windows="size", positive_rate="mean")
      .sort_index()
)

print("\nZero-packet windows by label and attack_id:")
if "packet_count" in df.columns:
    zero_packet = pd.to_numeric(df["packet_count"], errors="coerce").eq(0)
    print(pd.crosstab(
        [df["attack_id"], df["y"]],
        zero_packet,
        rownames=["attack_id", "y"],
        colnames=["packet_count == 0"],
        margins=True
    ))
    print("Total zero-packet windows:", int(zero_packet.sum()))

A1. y × attack_id
attack_id  T0_Benign  T4_DoS     All
y                                   
0              92324     533   92857
1                  0   71660   71660
All            92324   72193  164517
NOTE: attack_overlap_fraction is not present in this CSV. The 533 T4_DoS/y=0 cases from the earlier audit cannot be verified from this feature table alone. Do not relabel them from attack_id; verify the exact extractor version/event timestamps.

Top 15 longest runs by number of windows:
                                                   run_id mission_id attack_id  n_windows  n_positive  positive_rate  first_window  last_window  zero_packet_windows  span_seconds_approx
     run_0002_F1_afar_trace_1_C1_LOS_T0_Benign_ref3400_s2         F1 T0_Benign      27213           0       0.000000  1791659424.0 1791686636.0                26272              27213.0
       run_0107_F7_afar_trace_7_C3_NLOS_T4_DoS_ref3400_s2         F7    T4_DoS       7221        7205       0.997784  1791756146.0 179176

## B. Feature audit and fold-safe reduction

The earlier audit found 23 constant columns and multiple perfectly correlated pairs. Here we:
- list constants and exact duplicate columns for diagnostic purposes;
- exclude metadata and target columns from model inputs;
- during each LOMO fold, remove all-missing/constant columns and exact duplicates using **only the outer training data**.

The raw-vs-pruned comparison is an ablation, not a promise that fewer features will score better.

In [11]:
META_COLS = {
    "y", "run_id", "mission_id", "channel_id", "attack_id", "seed",
    "condition_group", "window_start_epoch_s", "window_size_s",
    "label", "label_binary", "window_label", "split", "fold",
    "source_file", "source_pcap", "pcap_path",
    "attack_overlap_ratio", "attack_overlap_fraction", "attack_overlap_seconds"
}
name_tokens_to_exclude = ("label", "overlap", "ground_truth", "groundtruth", "pcap_path", "source_path")

numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in META_COLS
    and not any(t in c.lower() for t in name_tokens_to_exclude)
]
if not numeric_cols:
    raise ValueError("Không tìm thấy numeric network features sau khi loại metadata.")

X_all = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

constant_cols = [
    c for c in numeric_cols
    if X_all[c].nunique(dropna=True) <= 1
]
print("Numeric candidate features:", len(numeric_cols))
print("Constant/all-missing columns across this dataset (diagnostic only):")
print(constant_cols)

def find_exact_duplicate_columns(frame, columns):
    # Hashing speeds up candidate discovery; Series.equals confirms exact equality.
    buckets = {}
    duplicates = []
    for col in columns:
        s = frame[col]
        h = hashlib.sha256(
            pd.util.hash_pandas_object(s, index=False).values.tobytes()
        ).hexdigest()
        matched = None
        for prior in buckets.get(h, []):
            if s.equals(frame[prior]):
                matched = prior
                break
        if matched is not None:
            duplicates.append((col, matched))
        else:
            buckets.setdefault(h, []).append(col)
    return duplicates

exact_dup_pairs_all = find_exact_duplicate_columns(X_all, numeric_cols)
print("\nExact duplicate column pairs (diagnostic across all rows; not used directly to select a fold):")
print(exact_dup_pairs_all if exact_dup_pairs_all else "None found")

# Explicitly verify suspected deterministic relationships.
suspected_pairs = [
    ("packet_count", "packets_per_second"),
    ("packet_count", "transport_udp_count"),
    ("packet_count", "ipproto_17_count"),
    ("total_bytes", "bytes_per_second"),
    ("unique_src_endpoints", "unique_5tuples"),
]
for a, b in suspected_pairs:
    if a in df.columns and b in df.columns:
        sa, sb = df[a], df[b]
        same = sa.eq(sb) | (sa.isna() & sb.isna())
        print(f"Exact row equality {a} == {b}: {100 * same.mean():.4f}%")

# High correlations are useful for redundancy screening, but are not proof of leakage.
if len(numeric_cols) >= 2:
    corr = X_all[numeric_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.stack().sort_values(ascending=False).head(20)
    print("\nTop 20 |correlation| pairs:")
    print(pairs.to_string())

def select_fold_features(X_train, mode):
    # X_train must be the outer training matrix only.
    usable = [c for c in X_train.columns if X_train[c].notna().any()]
    if mode == "raw":
        return usable, []
    if mode != "pruned":
        raise ValueError("mode must be 'raw' or 'pruned'")

    nonconstant = [c for c in usable if X_train[c].nunique(dropna=True) > 1]
    dropped = [c for c in usable if c not in nonconstant]
    duplicate_pairs = find_exact_duplicate_columns(X_train, nonconstant)
    duplicate_columns = {col for col, prior in duplicate_pairs}
    selected = [c for c in nonconstant if c not in duplicate_columns]
    dropped.extend(sorted(duplicate_columns))
    return selected, dropped

print("\nFeature QC is ready. Fold-wise pruning will be applied inside LOMO.")

Numeric candidate features: 57
Constant/all-missing columns across this dataset (diagnostic only):
['ipproto_1_count', 'ipproto_2_count', 'ipproto_6_count', 'ipproto_41_count', 'ipproto_47_count', 'ipproto_50_count', 'ipproto_58_count', 'ipproto_other_count', 'transport_tcp_count', 'transport_icmp_count', 'transport_other_count', 'transport_tcp_prop', 'transport_icmp_prop', 'transport_other_prop', 'ipproto_1_prop', 'ipproto_2_prop', 'ipproto_6_prop', 'ipproto_41_prop', 'ipproto_47_prop', 'ipproto_50_prop', 'ipproto_58_prop', 'ipproto_other_prop']

Exact duplicate column pairs (diagnostic across all rows; not used directly to select a fold):
[('wire_len_mean', 'captured_len_mean'), ('wire_len_std', 'captured_len_std'), ('wire_len_min', 'captured_len_min'), ('wire_len_max', 'captured_len_max'), ('wire_len_median', 'captured_len_median'), ('wire_len_q25', 'captured_len_q25'), ('wire_len_q75', 'captured_len_q75'), ('wire_len_q90', 'captured_len_q90'), ('ipproto_2_count', 'ipproto_1_count')

## C. Improved LOMO experiment

### Factors compared
- **Feature mode:** `raw` vs `pruned` (fold-wise only).
- **Training weighting:** `window_weighted` vs `equal_run_weighted`.
- **Models:** Logistic Regression, Extra Trees, HistGradientBoosting.

The equal-run weighting option gives every run the same total training weight, regardless of its window count. It is a sensitivity test for run-length imbalance, not a guaranteed improvement. Class weights are not added in this ablation so run weights remain interpretable.

No threshold tuning or hyperparameter search is done here. The predicted class uses the model default decision rule; probability-based metrics are also reported.

In [12]:
def make_estimator(model_name):
    if model_name == "LogisticRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=1.0, penalty="l2", solver="lbfgs",
                max_iter=2000, random_state=SEED
            ))
        ])
    if model_name == "ExtraTrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=100, max_features="sqrt", min_samples_leaf=2,
                class_weight=None, n_jobs=-1, random_state=SEED
            ))
        ])
    if model_name == "HistGradientBoosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                max_iter=100, max_leaf_nodes=15,
                l2_regularization=1.0, random_state=SEED
            ))
        ])
    raise KeyError(model_name)

def equal_run_weights(run_ids):
    # Mean weight is normalized to 1, so the regularization scale is comparable.
    s = pd.Series(np.asarray(run_ids, dtype=str))
    counts = s.map(s.value_counts()).to_numpy(dtype=float)
    w = 1.0 / counts
    return w / w.mean()

def score_vector(estimator, X):
    clf = estimator.named_steps["clf"]
    if hasattr(estimator, "predict_proba"):
        p = estimator.predict_proba(X)
        classes = list(clf.classes_)
        if 1 not in classes:
            raise ValueError(f"Model does not have positive class 1: {classes}")
        return np.asarray(p)[:, classes.index(1)]
    if hasattr(estimator, "decision_function"):
        z = np.asarray(estimator.decision_function(X))
        classes = list(clf.classes_)
        if z.ndim == 1:
            # sklearn's binary decision_function is oriented toward classes_[1].
            # Current labels should be [0, 1]; guard against a different class order.
            if classes != [0, 1]:
                raise ValueError(f"Unexpected classes_ order for binary decision_function: {classes}")
            return z
        return z[:, classes.index(1)]
    return estimator.predict(X).astype(float)

def calculate_metrics(y_true, y_pred, y_score, sample_weight=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    result = {
        "DoS_precision": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0, sample_weight=sample_weight
        ),
        "DoS_recall": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0, sample_weight=sample_weight
        ),
        "DoS_F1": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0, sample_weight=sample_weight
        ),
        "macro_F1": f1_score(
            y_true, y_pred, average="macro", labels=[0, 1],
            zero_division=0, sample_weight=sample_weight
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        "PR_AUC_average_precision": average_precision_score(
            y_true, y_score, sample_weight=sample_weight
        ) if np.any(y_true == 1) else np.nan,
        "ROC_AUC": roc_auc_score(
            y_true, y_score, sample_weight=sample_weight
        ) if len(np.unique(y_true)) == 2 else np.nan,
    }
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1], sample_weight=sample_weight
    ).ravel()
    result["FPR"] = fp / (fp + tn) if (fp + tn) else np.nan
    return result

MODEL_NAMES = ["LogisticRegression", "ExtraTrees", "HistGradientBoosting"]
FEATURE_MODES = ["raw", "pruned"]
TRAIN_WEIGHTING = ["window_weighted", "equal_run_weighted"]

metric_rows = []
prediction_frames = []
selected_feature_rows = []

missions = sorted(df["mission_id"].astype(str).unique())
if len(missions) < 3:
    warnings.warn(f"Chỉ có {len(missions)} mission; các kết luận LOMO sẽ rất không chắc chắn.")

for fold_no, heldout_mission in enumerate(missions, start=1):
    train_df = df.loc[df["mission_id"].astype(str) != heldout_mission].copy()
    test_df = df.loc[df["mission_id"].astype(str) == heldout_mission].copy()

    if train_df.empty or test_df.empty:
        continue
    y_train = train_df["y"].to_numpy(dtype=int)
    y_test = test_df["y"].to_numpy(dtype=int)
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        print("SKIP fold; train/test thiếu một lớp:", heldout_mission)
        continue

    X_train_all = train_df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    X_test_all = test_df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    test_w = equal_run_weights(test_df["run_id"].astype(str))

    print(
        f"\nLOMO fold {fold_no}/{len(missions)} | held-out={heldout_mission} | "
        f"train={len(train_df):,} test={len(test_df):,} | "
        f"train_runs={train_df['run_id'].nunique()} test_runs={test_df['run_id'].nunique()}"
    )

    for feature_mode in FEATURE_MODES:
        fold_features, dropped_features = select_fold_features(X_train_all, feature_mode)
        if not fold_features:
            print("No usable features; skip:", heldout_mission, feature_mode)
            continue
        X_train = X_train_all[fold_features]
        X_test = X_test_all[fold_features]

        selected_feature_rows.append({
            "held_out_mission": heldout_mission,
            "feature_mode": feature_mode,
            "n_selected_features": len(fold_features),
            "selected_features": json.dumps(fold_features),
            "dropped_features": json.dumps(dropped_features)
        })

        for weighting in TRAIN_WEIGHTING:
            for model_name in MODEL_NAMES:
                est = make_estimator(model_name)
                try:
                    if weighting == "equal_run_weighted":
                        train_w = equal_run_weights(train_df["run_id"].astype(str))
                        est.fit(X_train, y_train, clf__sample_weight=train_w)
                    else:
                        est.fit(X_train, y_train)

                    y_pred = est.predict(X_test).astype(int)
                    y_score = score_vector(est, X_test)

                    window_metrics = calculate_metrics(y_test, y_pred, y_score)
                    runbalanced_metrics = calculate_metrics(
                        y_test, y_pred, y_score, sample_weight=test_w
                    )

                    row = {
                        "experiment": "LOMO_1s",
                        "held_out_mission": heldout_mission,
                        "model": model_name,
                        "feature_mode": feature_mode,
                        "train_weighting": weighting,
                        "n_train_windows": len(train_df),
                        "n_test_windows": len(test_df),
                        "n_train_runs": train_df["run_id"].nunique(),
                        "n_test_runs": test_df["run_id"].nunique(),
                        "n_features": len(fold_features),
                        "test_positive_rate": float(y_test.mean()),
                    }
                    row.update({f"{k}_window": v for k, v in window_metrics.items()})
                    row.update({f"{k}_run_balanced": v for k, v in runbalanced_metrics.items()})
                    metric_rows.append(row)

                    # Keep detailed OOF predictions only for pruned features to limit file size.
                    if feature_mode == "pruned":
                        pred_frame = test_df[
                            ["run_id", "mission_id", "attack_id", "window_start_epoch_s", "y"]
                        ].copy()
                        pred_frame = pred_frame.rename(columns={"y": "y_true"})
                        pred_frame["held_out_mission"] = heldout_mission
                        pred_frame["model"] = model_name
                        pred_frame["train_weighting"] = weighting
                        pred_frame["y_pred"] = y_pred
                        pred_frame["y_score"] = y_score
                        prediction_frames.append(pred_frame)

                    print(
                        f"  {feature_mode:6s} | {weighting:18s} | {model_name:22s} | "
                        f"F1 window={window_metrics['DoS_F1']:.3f} | "
                        f"F1 run-balanced={runbalanced_metrics['DoS_F1']:.3f} | "
                        f"AUC={window_metrics['ROC_AUC']:.3f}"
                    )
                except Exception as exc:
                    metric_rows.append({
                        "experiment": "LOMO_1s",
                        "held_out_mission": heldout_mission,
                        "model": model_name,
                        "feature_mode": feature_mode,
                        "train_weighting": weighting,
                        "error": f"{type(exc).__name__}: {exc}"
                    })
                    print(f"ERROR {heldout_mission}/{feature_mode}/{weighting}/{model_name}: {exc}")

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
selected_features_df = pd.DataFrame(selected_feature_rows)

metrics_path = OUT_DIR / "lomo_metrics_per_mission.csv"
predictions_path = OUT_DIR / "lomo_oof_predictions_pruned.csv"
features_path = OUT_DIR / "lomo_selected_features.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
selected_features_df.to_csv(features_path, index=False)

print("\nSaved:")
print(metrics_path)
print(predictions_path)
print(features_path)


LOMO fold 1/3 | held-out=F1 | train=111,578 test=52,939 | train_runs=80 test_runs=40
  raw    | window_weighted    | LogisticRegression     | F1 window=0.275 | F1 run-balanced=0.513 | AUC=0.185
  raw    | window_weighted    | ExtraTrees             | F1 window=0.235 | F1 run-balanced=0.461 | AUC=0.176
  raw    | window_weighted    | HistGradientBoosting   | F1 window=0.245 | F1 run-balanced=0.480 | AUC=0.179
  raw    | equal_run_weighted | LogisticRegression     | F1 window=0.269 | F1 run-balanced=0.507 | AUC=0.184
  raw    | equal_run_weighted | ExtraTrees             | F1 window=0.228 | F1 run-balanced=0.453 | AUC=0.186
  raw    | equal_run_weighted | HistGradientBoosting   | F1 window=0.243 | F1 run-balanced=0.477 | AUC=0.207
  pruned | window_weighted    | LogisticRegression     | F1 window=0.273 | F1 run-balanced=0.512 | AUC=0.191
  pruned | window_weighted    | ExtraTrees             | F1 window=0.234 | F1 run-balanced=0.458 | AUC=0.188
  pruned | window_weighted    | HistGradie

## D. Summarize the results

Do not select a “winner” only from the mean score. With only three outer missions, show all three per-mission results and their standard deviation. Prefer a model that is stable across F1/F4/F7 over one that scores very well on one mission but fails on another.

- `*_window`: every test window has equal evaluation weight.
- `*_run_balanced`: test samples are weighted inversely to the number of windows in their run, so each run has equal total weight.
- Compare `raw` vs `pruned` to test whether constants and exact duplicates matter.
- Compare `window_weighted` vs `equal_run_weighted` to test whether long runs dominate fitting.

These are still preliminary results: no nested hyperparameter tuning or operating-threshold selection has been performed.

In [13]:
valid_metrics = metrics_df.loc[metrics_df.get("error", pd.Series(index=metrics_df.index, dtype=object)).isna()].copy()     if not metrics_df.empty else pd.DataFrame()

if not valid_metrics.empty:
    metric_cols = [
        "DoS_F1_window", "balanced_accuracy_window", "ROC_AUC_window",
        "PR_AUC_average_precision_window", "FPR_window",
        "DoS_F1_run_balanced", "balanced_accuracy_run_balanced",
        "ROC_AUC_run_balanced", "PR_AUC_average_precision_run_balanced",
        "FPR_run_balanced"
    ]
    available_cols = [c for c in metric_cols if c in valid_metrics.columns]
    summary = (
        valid_metrics.groupby(["model", "feature_mode", "train_weighting"])[available_cols]
        .agg(["mean", "std"])
        .reset_index()
    )
    summary.columns = [
        "_".join(str(part) for part in col if str(part))
        if isinstance(col, tuple) else str(col)
        for col in summary.columns
    ]
    summary_path = OUT_DIR / "lomo_summary_mean_std.csv"
    summary.to_csv(summary_path, index=False)

    print("Per-mission results:")
    display(valid_metrics.sort_values(
        ["model", "feature_mode", "train_weighting", "held_out_mission"]
    ))
    print("\nMean ± standard deviation over the 3 held-out missions:")
    display(summary)
    print("\nSaved summary:", summary_path)
else:
    print("No valid metrics. Check error rows in metrics_df.")

print("\nOutput directory:", OUT_DIR)
print("Files:", [p.name for p in OUT_DIR.iterdir() if p.is_file()])

Per-mission results:


,experiment,held_out_mission,model,feature_mode,train_weighting,n_train_windows,n_test_windows,n_train_runs,n_test_runs,n_features,test_positive_rate,DoS_precision_window,DoS_recall_window,DoS_F1_window,macro_F1_window,balanced_accuracy_window,PR_AUC_average_precision_window,ROC_AUC_window,FPR_window,DoS_precision_run_balanced,DoS_recall_run_balanced,DoS_F1_run_balanced,macro_F1_run_balanced,balanced_accuracy_run_balanced,PR_AUC_average_precision_run_balanced,ROC_AUC_run_balanced,FPR_run_balanced
10,LOMO_1s,F1,ExtraTrees,pruned,equal_run_weighted,111578,52939,80,40,23,0.252177,0.154829,0.435805,0.228484,0.256733,0.316794,0.156264,0.194985,0.802218,0.479379,0.426807,0.451568,0.482968,0.484515,0.505654,0.473986,0.457778
22,LOMO_1s,F4,ExtraTrees,pruned,equal_run_weighted,105779,58738,80,40,23,0.498008,0.514150,0.572645,0.541823,0.516347,0.517907,0.570176,0.536795,0.536831,0.512080,0.566052,0.537715,0.515828,0.517165,0.568903,0.538501,0.531722
34,LOMO_1s,F7,ExtraTrees,pruned,equal_run_weighted,111677,52840,80,40,23,0.549924,0.527146,0.384266,0.444506,0.470558,0.481554,0.588705,0.456290,0.421159,0.490852,0.433489,0.460391,0.495038,0.496731,0.561940,0.518684,0.440027
7,LOMO_1s,F1,ExtraTrees,pruned,window_weighted,111578,52939,80,40,23,0.252177,0.158112,0.450412,0.234060,0.255971,0.320839,0.155896,0.188441,0.808735,0.477265,0.441107,0.458474,0.481239,0.481983,0.502379,0.475769,0.477141
19,LOMO_1s,F4,ExtraTrees,pruned,window_weighted,105779,58738,80,40,23,0.498008,0.525389,0.581499,0.552022,0.528840,0.530185,0.564288,0.544705,0.521129,0.522302,0.573866,0.546871,0.527058,0.528213,0.563671,0.545775,0.517440
31,LOMO_1s,F7,ExtraTrees,pruned,window_weighted,111677,52840,80,40,23,0.549924,0.542695,0.387570,0.452198,0.481907,0.494264,0.593241,0.463853,0.399041,0.505498,0.436502,0.468473,0.507079,0.509315,0.566943,0.527547,0.417871
4,LOMO_1s,F1,ExtraTrees,raw,equal_run_weighted,111578,52939,80,40,57,0.252177,0.154838,0.433408,0.228164,0.259241,0.317831,0.155916,0.186280,0.797747,0.484928,0.425495,0.453271,0.487667,0.489577,0.504135,0.478535,0.446341
16,LOMO_1s,F4,ExtraTrees,raw,equal_run_weighted,105779,58738,80,40,57,0.498008,0.515701,0.558594,0.536292,0.518256,0.519089,0.574089,0.539706,0.520416,0.515386,0.555960,0.534905,0.519548,0.520292,0.573455,0.542763,0.515375
28,LOMO_1s,F7,ExtraTrees,raw,equal_run_weighted,111677,52840,80,40,57,0.549924,0.531339,0.372255,0.437793,0.472007,0.485535,0.583101,0.452952,0.401186,0.492614,0.417063,0.451701,0.495431,0.498342,0.555048,0.513842,0.420380
1,LOMO_1s,F1,ExtraTrees,raw,window_weighted,111578,52939,80,40,57,0.252177,0.158948,0.452210,0.235218,0.257764,0.322659,0.156060,0.176220,0.806891,0.480435,0.443692,0.461333,0.484154,0.484906,0.502719,0.479012,0.473880



Mean ± standard deviation over the 3 held-out missions:


,model,feature_mode,train_weighting,DoS_F1_window_mean,DoS_F1_window_std,balanced_accuracy_window_mean,balanced_accuracy_window_std,ROC_AUC_window_mean,ROC_AUC_window_std,PR_AUC_average_precision_window_mean,PR_AUC_average_precision_window_std,FPR_window_mean,FPR_window_std,DoS_F1_run_balanced_mean,DoS_F1_run_balanced_std,balanced_accuracy_run_balanced_mean,balanced_accuracy_run_balanced_std,ROC_AUC_run_balanced_mean,ROC_AUC_run_balanced_std,PR_AUC_average_precision_run_balanced_mean,PR_AUC_average_precision_run_balanced_std,FPR_run_balanced_mean,FPR_run_balanced_std
0,ExtraTrees,pruned,equal_run_weighted,0.404938,0.160373,0.438751,0.107171,0.396024,0.178697,0.438381,0.244497,0.586736,0.195370,0.483225,0.047396,0.499470,0.016497,0.510390,0.033048,0.545499,0.034682,0.476509,0.048632
1,ExtraTrees,pruned,window_weighted,0.412760,0.162608,0.448429,0.111947,0.399000,0.186777,0.437808,0.244572,0.576302,0.210345,0.491273,0.048408,0.506504,0.023243,0.516364,0.036318,0.544331,0.036369,0.470817,0.050085
2,ExtraTrees,raw,equal_run_weighted,0.400750,0.157369,0.440818,0.107823,0.392979,0.184187,0.437702,0.244075,0.573116,0.203465,0.479959,0.047591,0.502737,0.015822,0.511713,0.032167,0.544213,0.035908,0.460699,0.049098
3,ExtraTrees,raw,window_weighted,0.408393,0.158827,0.451406,0.113168,0.396347,0.196349,0.439817,0.245790,0.560238,0.221104,0.486875,0.049070,0.508901,0.024429,0.518178,0.038115,0.546166,0.038741,0.455083,0.048822
4,HistGradientBoosting,pruned,equal_run_weighted,0.416610,0.161343,0.436329,0.095398,0.393534,0.168166,0.447842,0.216730,0.622056,0.158137,0.498039,0.050995,0.498973,0.022047,0.510954,0.018397,0.550860,0.015139,0.507932,0.061467
5,HistGradientBoosting,pruned,window_weighted,0.435044,0.171175,0.450222,0.105516,0.393197,0.190659,0.433392,0.240199,0.616131,0.164866,0.518503,0.047935,0.514284,0.020467,0.520714,0.023928,0.540265,0.031944,0.503100,0.045857
6,HistGradientBoosting,raw,equal_run_weighted,0.416610,0.161343,0.436329,0.095398,0.393534,0.168166,0.447842,0.216730,0.622056,0.158137,0.498039,0.050995,0.498973,0.022047,0.510954,0.018397,0.550860,0.015139,0.507932,0.061467
7,HistGradientBoosting,raw,window_weighted,0.435044,0.171175,0.450222,0.105516,0.393197,0.190659,0.433392,0.240199,0.616131,0.164866,0.518503,0.047935,0.514284,0.020467,0.520714,0.023928,0.540265,0.031944,0.503100,0.045857
8,LogisticRegression,pruned,equal_run_weighted,0.320304,0.153206,0.440539,0.073783,0.365957,0.174465,0.420177,0.222478,0.505218,0.324541,0.398369,0.188600,0.489507,0.019951,0.485609,0.069879,0.519911,0.056147,0.405191,0.190940
9,LogisticRegression,pruned,window_weighted,0.332627,0.189606,0.459183,0.086099,0.381007,0.186311,0.428570,0.229245,0.492254,0.359389,0.408107,0.222080,0.507499,0.020956,0.501787,0.070363,0.529311,0.055000,0.390842,0.238122



Saved summary: /kaggle/working/uav_c2_research_improved/lomo_summary_mean_std.csv

Output directory: /kaggle/working/uav_c2_research_improved
Files: ['lomo_selected_features.csv', 'lomo_summary_mean_std.csv', 'run_audit.csv', 'lomo_oof_predictions_pruned.csv', 'lomo_metrics_per_mission.csv']


## E. What to conclude in the paper

1. **Labeling:** describe `y` as a window-level label using the predefined attack-event overlap rule, if the CSV provenance confirms this. Explain why a DoS run can contain `y=0` windows.
2. **Data quality:** report no missing or infinite values and no exact duplicate rows based on the audit, while separately reporting constant/redundant features and the long-run imbalance.
3. **Primary evaluation:** LOMO, three mission-specific folds. Present all folds; do not hide a poor mission behind one pooled score.
4. **Robustness ablation:** compare raw vs pruned features and window- vs run-balanced training. Treat any increase as evidence only if it is consistent across missions.
5. **Secondary experiment:** add grouped validation by `run_id` or `condition_group` to measure generalization across unseen runs within known missions. This is a different question from LOMO and should be reported separately.
6. **Threshold calibration:** only after baseline and data audit are stable, select a threshold using training-only inner validation; apply that fixed threshold to the outer held-out mission.
7. **Limits:** three missions are only three outer test groups. Report this as a limitation and avoid overclaiming generalization to all UAV C2 environments.

### Avoid these shortcuts
- Do not set all rows from `T4_DoS` to label 1 without checking the event-time rule.
- Do not fit scalers, feature selectors, threshold tuners or hyperparameter search on the held-out mission.
- Do not use row-random train/test split as the primary result.
- Do not call high feature-label correlation or single-feature AUC “leakage” without tracing the label generation and split.